In [3]:
from pathlib import Path
import pandas as pd

# -------------------------------------------------------
# Input and output files
# -------------------------------------------------------
DATA_DIR = Path.cwd()

file_1 = "fit_fraction_selected_results_388.csv"
file_2 = "fit_fraction_selected_results_410.csv"

output_file = "fit_fraction_selected_results_difference.csv"

# -------------------------------------------------------
# Load the CSV files
# -------------------------------------------------------
df_1 = pd.read_csv(file_1)
df_2 = pd.read_csv(file_2)

# Columns used to identify corresponding rows
key_columns = [
    "group",
    "bin",
    "line",
    "parameter",
]

# Columns for which the differences will be calculated
numeric_columns = [
    "value",
    "err_lower",
    "err_upper",
    "lower_limit",
    "upper_limit",
]

# -------------------------------------------------------
# Validate the required columns
# -------------------------------------------------------
required_columns = key_columns + numeric_columns

for filename, dataframe in [(file_1, df_1), (file_2, df_2)]:
    missing_columns = [
        column
        for column in required_columns
        if column not in dataframe.columns
    ]

    if missing_columns:
        raise ValueError(
            f"{filename.name} is missing these columns: "
            f"{missing_columns}"
        )

# Each combination of identifiers should occur only once
if df_1.duplicated(key_columns).any():
    raise ValueError(
        f"{file_1.name} contains duplicated combinations of "
        f"{key_columns}"
    )

if df_2.duplicated(key_columns).any():
    raise ValueError(
        f"{file_2.name} contains duplicated combinations of "
        f"{key_columns}"
    )

# -------------------------------------------------------
# Match corresponding rows
# -------------------------------------------------------
comparison = df_1[
    key_columns + numeric_columns
].merge(
    df_2[key_columns + numeric_columns],
    on=key_columns,
    how="outer",
    suffixes=("_1", "_2"),
    indicator=True,
    validate="one_to_one",
)

# Check whether one file contains rows absent from the other
unmatched = comparison.loc[
    comparison["_merge"] != "both",
    key_columns + ["_merge"],
]

if not unmatched.empty:
    raise ValueError(
        "The files do not contain exactly the same samples.\n"
        f"Unmatched rows:\n{unmatched.to_string(index=False)}"
    )

# -------------------------------------------------------
# Calculate differences
#
# difference = file 1 - file 2
# -------------------------------------------------------
difference = comparison[key_columns].copy()

for column in numeric_columns:
    difference[column] = (
        comparison[f"{column}_1"]
        - comparison[f"{column}_2"]
    )

# -------------------------------------------------------
# Export the results
# -------------------------------------------------------
difference.to_csv(output_file, index=False)

print(f"Results saved to: {output_file}")
display(difference.head(10))

Results saved to: fit_fraction_selected_results_difference.csv


,group,bin,line,parameter,value,err_lower,err_upper,lower_limit,upper_limit
0,KPNO,0,H,m,0.000698,-0.002442,0.001866,0.003140,0.002563
1,KPNO,0,H,noise,-0.000035,0.000160,-0.000116,-0.000195,-0.000151
2,KPNO,0,H,r0,-0.003470,-0.000021,-0.000314,-0.003449,-0.003783
3,KPNO,0,H,s0,-0.101863,0.000831,-0.044383,-0.102695,-0.146246
4,KPNO,0,H,sig2,-0.011182,-0.001215,-0.021024,-0.009967,-0.032206
5,KPNO,0,N,m,-0.010705,-0.010786,0.000543,0.000081,-0.010162
6,KPNO,0,N,noise,0.000478,-0.000141,-0.000645,0.000619,-0.000167
7,KPNO,0,N,r0,-0.001476,-0.000206,-0.000882,-0.001270,-0.002358
8,KPNO,0,N,s0,-0.056857,-0.009595,-0.096004,-0.047262,-0.152861
9,KPNO,0,N,sig2,0.099306,-0.007913,-0.094164,0.107220,0.005142
